In [15]:
# ------------------------------
# Simple LSTM Model
# ------------------------------
import torch
import torch.nn as nn

class SimpleLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()

        # LSTM expects (batch, seq_len, input_size)
        # We treat each feature as one timestep
        self.lstm = nn.LSTM(
            input_size=1,           # each feature is a scalar
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x shape: (batch, features)
        x = x.unsqueeze(-1)        # (batch, seq_len=features, 1)

        lstm_out, (h_n, _) = self.lstm(x)

        # Use last layer hidden state
        final_hidden = h_n[-1]     # (batch, hidden_dim)

        out = self.fc(final_hidden)
        return out


In [16]:
# ------------------------------
# Load & Prepare Dataset (Year kept only for split/eval)
# ------------------------------
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv("corn_dataset_final.csv")

# --- Config ---
TRAIN_YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
TEST_YEARS  = [2020,2021,2022]
TARGET_COL  = "YIELD"

# Filter train/test years
train_df_all = df[df["year"].isin(TRAIN_YEARS)].copy()
test_df      = df[df["year"].isin(TEST_YEARS)].copy()

# ------------------------------
# Encode states 
# ------------------------------
states = sorted(train_df_all["state_name"].unique())
state_to_idx = {s: i for i, s in enumerate(states)}
train_df_all["state_idx"] = train_df_all["state_name"].map(state_to_idx)
test_df["state_idx"]      = test_df["state_name"].map(state_to_idx)

# ------------------------------
# Feature columns
# ------------------------------
# Drop year entirely from features
feature_cols = [
    "ppt (mm)_AVG", "tmax (degrees C)_AVG", "tmean (degrees C)_AVG", "tmin (degrees C)_AVG",  # climate
    "ssm_AVG", "susm_AVG",  # soil
    "EVI_AVG", "GCI_AVG", "NDWI_AVG", "NDVI_AVG",  # satellite
    "AWC", "SOM", "CEC", "state_idx",  # static
    "HIST_AVG_YIELD"  # new historical average yield feature
]

# Columns to scale (all except target & year)
scaler_X = StandardScaler()
train_df_all[feature_cols] = scaler_X.fit_transform(train_df_all[feature_cols])
test_df[feature_cols]      = scaler_X.transform(test_df[feature_cols])

# Scale target
scaler_y = StandardScaler()
train_df_all["YIELD_STD"] = scaler_y.fit_transform(train_df_all[[TARGET_COL]])
test_df["YIELD_STD"]      = scaler_y.transform(test_df[[TARGET_COL]])

# ------------------------------
# Dataset class (flattened for single cross-attention)
# ------------------------------
class FlattenedCornDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.target_col = "YIELD_STD"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        X = torch.tensor(row[self.feature_cols].to_numpy(dtype=np.float32), dtype=torch.float32)
        y = torch.tensor(float(row[self.target_col]), dtype=torch.float32).unsqueeze(0)
        return X, y

# ------------------------------
# Create datasets & loaders
# ------------------------------
full_train_dataset = FlattenedCornDataset(train_df_all)
test_dataset       = FlattenedCornDataset(test_df)

train_idx, val_idx = train_test_split(
    range(len(full_train_dataset)),
    test_size=0.2,
    random_state=42
)

train_dataset = torch.utils.data.Subset(full_train_dataset, train_idx)
val_dataset   = torch.utils.data.Subset(full_train_dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

# ------------------------------
# Input info
# ------------------------------
sample_X, sample_y = train_dataset[0]

print(f"Flattened feature vector: {sample_X.shape[0]}")
print(f"Target shape: {sample_y.shape}")

Flattened feature vector: 15
Target shape: torch.Size([1])


In [17]:
class EarlyStopping:
    def __init__(self, patience=100, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1

        if self.counter >= self.patience:
            self.stop = True

In [18]:
import torch.optim as optim
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = len(feature_cols)

model = SimpleLSTMModel(
    input_dim=input_dim,
    hidden_dim=128,
    num_layers=2,
    dropout=0.2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

early_stopper = EarlyStopping(patience=15, min_delta=0.001)

best_val_mse = float("inf")
best_model_path = "best_lstm_model.pth"

for epoch in range(200):
    model.train()
    train_preds, train_truths = [], []

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        train_preds.append(pred.detach().cpu().numpy())
        train_truths.append(y.detach().cpu().numpy())

    train_preds = np.vstack(train_preds)
    train_truths = np.vstack(train_truths)
    train_mse = mean_squared_error(train_truths, train_preds)
    train_r2 = r2_score(train_truths, train_preds)

    train_preds_real = scaler_y.inverse_transform(train_preds)
    train_truths_real = scaler_y.inverse_transform(train_truths)
    train_r2_real = r2_score(train_truths_real, train_preds_real)

    # ------------------------------
    # Validation
    # ------------------------------
    model.eval()
    val_preds, val_truths = [], []

    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            val_preds.append(out.cpu().numpy())
            val_truths.append(y.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_truths = np.vstack(val_truths)
    val_mse = mean_squared_error(val_truths, val_preds)
    val_r2 = r2_score(val_truths, val_preds)

    val_preds_real = scaler_y.inverse_transform(val_preds)
    val_truths_real = scaler_y.inverse_transform(val_truths)
    val_r2_real = r2_score(val_truths_real, val_preds_real)

    print(
        f"[Epoch {epoch+1}] "
        f"Train MSE(scaled): {train_mse:.4f}, Train R2(scaled): {train_r2:.4f}, Train R2(real): {train_r2_real:.4f} | "
        f"Val MSE(scaled): {val_mse:.4f}, Val R2(scaled): {val_r2:.4f}, Val R2(real): {val_r2_real:.4f}"
    )

    # ------------------------------
    # Save best model
    # ------------------------------
    if val_mse < best_val_mse:
        best_val_mse = val_mse
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model at epoch {epoch+1} with Val MSE: {val_mse:.4f}")


    # ------------------------------
    # Early stopping
    # ------------------------------
    early_stopper(val_mse)
    if early_stopper.stop:
        print(f"Stopping early at epoch {epoch+1}")
        break

[Epoch 1] Train MSE(scaled): 0.4212, Train R2(scaled): 0.5823, Train R2(real): 0.5823 | Val MSE(scaled): 0.2917, Val R2(scaled): 0.6979, Val R2(real): 0.6979
Saved new best model at epoch 1 with Val MSE: 0.2917
[Epoch 2] Train MSE(scaled): 0.2717, Train R2(scaled): 0.7305, Train R2(real): 0.7305 | Val MSE(scaled): 0.2633, Val R2(scaled): 0.7273, Val R2(real): 0.7273
Saved new best model at epoch 2 with Val MSE: 0.2633
[Epoch 3] Train MSE(scaled): 0.2626, Train R2(scaled): 0.7396, Train R2(real): 0.7396 | Val MSE(scaled): 0.2611, Val R2(scaled): 0.7296, Val R2(real): 0.7296
Saved new best model at epoch 3 with Val MSE: 0.2611
[Epoch 4] Train MSE(scaled): 0.2612, Train R2(scaled): 0.7409, Train R2(real): 0.7409 | Val MSE(scaled): 0.2634, Val R2(scaled): 0.7272, Val R2(real): 0.7272
[Epoch 5] Train MSE(scaled): 0.2586, Train R2(scaled): 0.7436, Train R2(real): 0.7436 | Val MSE(scaled): 0.2517, Val R2(scaled): 0.7393, Val R2(real): 0.7393
Saved new best model at epoch 5 with Val MSE: 0.251

In [21]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# ------------------------------
# Load Best LSTM Model
# ------------------------------
model = SimpleLSTMModel(
    input_dim=len(feature_cols),
    hidden_dim=128,
    num_layers=2,
    dropout=0.2
).to(device)

checkpoint_path = "best_lstm_model.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

# Make sure your test_df still has the original 'year' column for splitting
YEARS = [2022]

for year in YEARS:
    # Filter rows for the current year (original, unscaled year column)
    year_df = test_df[test_df["year"] == year].reset_index(drop=True)
    
    if len(year_df) == 0:
        print(f"No data for year {year}")
        continue

    # Use the flattened dataset class
    year_dataset = FlattenedCornDataset(year_df)
    year_loader  = DataLoader(year_dataset, batch_size=32, shuffle=False)

    all_preds_scaled, all_truths_scaled = [], []

    with torch.no_grad():
        for X, y in year_loader:
            # Move to device
            X, y = X.to(device), y.to(device)
            # Forward pass
            preds = model(X)  # model expects a single input vector now
            # Collect results
            all_preds_scaled.append(preds.cpu().numpy())
            all_truths_scaled.append(y.cpu().numpy())

    all_preds_scaled  = np.vstack(all_preds_scaled)
    all_truths_scaled = np.vstack(all_truths_scaled)

    # Convert scaled predictions back to real yield
    all_preds_real  = scaler_y.inverse_transform(all_preds_scaled)
    all_truths_real = scaler_y.inverse_transform(all_truths_scaled)

    # Compute metrics
    mse_scaled = mean_squared_error(all_truths_scaled, all_preds_scaled)
    r2_scaled  = r2_score(all_truths_scaled, all_preds_scaled)
    mse_real   = mean_squared_error(all_truths_real, all_preds_real)
    r2_real    = r2_score(all_truths_real, all_preds_real)

    print(f"--- Year {year} ---")
    print(f"MSE (scaled): {mse_scaled:.4f}, R2 (scaled): {r2_scaled:.4f}")
    print(f"MSE (real):   {mse_real:.4f}, R2 (real):   {r2_real:.4f}")


--- Year 2022 ---
MSE (scaled): 0.4565, R2 (scaled): 0.6742
MSE (real):   580.2980, R2 (real):   0.6742
